# Apple Orchard AI v3: retraining on real orchard photos

**Why:** model v2 was trained on PlantVillage (clean lab photos). In a field test on 9 photos from our orchards in Himachal Pradesh, it gave confident wrong answers on real field photos.

**What this notebook does:**
1. Loads **Plant Pathology 2021** (about 18,600 real orchard photos of apple leaves)
2. Keeps photos with exactly one label (drops "complex" and mixed labels)
3. Trains the **same model as v2** (torchvision EfficientNet-B0, 224x224) so the app needs only small changes
4. Tests it on photos it never saw, and checks how often wrong answers get past the app's 60% confidence threshold
5. (Optional) Re-runs the 9 orchard photos with **v2 and v3 side by side**

**Before running:** Settings → Accelerator → **GPU**, Internet → **On**. Add the competition data **Plant Pathology 2021 - FGVC8** as input.

## 1. Setup

In [ ]:
import os, glob, json, time, random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("EPOCHS", 12))        # 12 passes over the training photos
BATCH = int(os.environ.get("BATCH", 64))
IMG = 224                                         # same size as the app uses
THRESHOLD = 0.60                                  # the app's confidence threshold
USE_PRETRAINED = os.environ.get("USE_PRETRAINED", "1") == "1"   # start from ImageNet weights
INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
WORK = Path(os.environ.get("WORK", "/kaggle/working"))
RESIZED = Path(os.environ.get("RESIZED", "/tmp/resized"))
WORK.mkdir(parents=True, exist_ok=True); RESIZED.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
if device.type != "cuda":
    print("WARNING: no GPU found. Turn on GPU in Settings → Accelerator, or training will be very slow.")

## 2. Find the dataset and keep single-label photos

Class names match the keys in the app's `src/diseases.json`. Frogeye leaf spot is the leaf stage of black rot (same fungus), so it uses the existing **Black Rot** entry. **Powdery mildew** is new.

In [ ]:
csv_hits = [p for p in glob.glob(f"{INPUT_ROOT}/**/train.csv", recursive=True)
            if os.path.isdir(os.path.join(os.path.dirname(p), "train_images"))]
assert csv_hits, "Couldn't find train.csv next to a train_images folder. Did you add 'Plant Pathology 2021 - FGVC8' as input?"
DATA_DIR = os.path.dirname(csv_hits[0])
print("Data folder:", DATA_DIR)

LABEL_MAP = {
    "scab": "Apple___Apple_scab",
    "frog_eye_leaf_spot": "Apple___Black_rot",
    "rust": "Apple___Cedar_apple_rust",
    "healthy": "Apple___healthy",
    "powdery_mildew": "Apple___Powdery_mildew",
}
CLASS_NAMES = list(LABEL_MAP.values())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print("All photos:", len(df))
df = df[df["labels"].isin(LABEL_MAP.keys())].copy()        # one label only, no "complex"
df["cls"] = df["labels"].map(LABEL_MAP)
df["y"] = df["cls"].map(CLASS_TO_IDX)
print("Kept (single label):", len(df))
print(df["cls"].value_counts())

## 3. Shrink the photos once (makes training much faster)

The originals are about 4000x2700 pixels. We shrink them to 256x256 once, the same way the app squashes photos to a square.

In [ ]:
def shrink(name):
    out = RESIZED / name
    if out.exists():
        return
    with Image.open(os.path.join(DATA_DIR, "train_images", name)) as im:
        im.draft("RGB", (512, 512))          # fast JPEG decode at lower resolution
        im.convert("RGB").resize((256, 256), Image.BILINEAR).save(out, quality=92)

t0 = time.time()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    list(ex.map(shrink, df["image"].tolist(), chunksize=64))
print(f"Shrunk {len(df)} photos in {time.time()-t0:.0f}s")

## 4. Split: 80% train, 10% validation (pick best epoch), 10% test (final score)

In [ ]:
train_df, rest = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.5, stratify=rest["y"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = T.Compose([
    T.RandomResizedCrop(IMG, scale=(0.6, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),       # different light, like real orchards
    T.ToTensor(), NORM,
])
eval_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), NORM])   # identical to the app

class Leaves(Dataset):
    def __init__(self, frame, tf):
        self.names, self.ys, self.tf = frame["image"].tolist(), frame["y"].tolist(), tf
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        img = Image.open(RESIZED / self.names[i]).convert("RGB")
        return self.tf(img), self.ys[i]

workers = min(4, os.cpu_count())
train_dl = DataLoader(Leaves(train_df, train_tf), BATCH, shuffle=True, num_workers=workers, pin_memory=True, drop_last=True)
val_dl = DataLoader(Leaves(val_df, eval_tf), BATCH, num_workers=workers, pin_memory=True)
test_dl = DataLoader(Leaves(test_df, eval_tf), BATCH, num_workers=workers, pin_memory=True)

## 5. Model: same as v2 (EfficientNet-B0), new last layer with 5 classes

In [ ]:
def build_model(n_classes, pretrained):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    return m

model = build_model(len(CLASS_NAMES), USE_PRETRAINED).to(device)

# Rarer classes (like powdery mildew) count a bit more, so the model doesn't ignore them
counts = np.bincount(train_df["y"], minlength=len(CLASS_NAMES))
weights = torch.tensor((counts.sum() / counts) / (counts.sum() / counts).mean(), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=EPOCHS * len(train_dl), pct_start=0.15)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

## 6. Train (keeps the epoch with the best validation score)

In [ ]:
def run_eval(dl):
    model.eval(); probs, ys = [], []
    with torch.no_grad(), torch.autocast(device.type, enabled=device.type == "cuda"):
        for x, y in dl:
            probs.append(torch.softmax(model(x.to(device)).float(), 1).cpu()); ys.append(y)
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

MODEL_PATH = WORK / "apple_disease_model_v3.pth"
history, best_f1 = [], -1
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); total = 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item()
    p, yv = run_eval(val_dl)
    acc, f1 = (p.argmax(1) == yv).mean(), f1_score(yv, p.argmax(1), average="macro")
    history.append({"epoch": epoch, "train_loss": total / len(train_dl), "val_acc": acc, "val_macro_f1": f1})
    saved = ""
    if f1 > best_f1:
        best_f1 = f1; torch.save(model.state_dict(), MODEL_PATH); saved = "  <- best so far, saved"
    print(f"Epoch {epoch:2d}  loss {total/len(train_dl):.3f}  val acc {acc:.3f}  val F1 {f1:.3f}  ({time.time()-t0:.0f}s){saved}")

## 7. Final test on photos the model never saw

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
p, yt = run_eval(test_dl)
pred, conf = p.argmax(1), p.max(1)
short = ["Scab", "Black rot / frogeye", "Rust", "Healthy", "Powdery mildew"]
print(f"Test accuracy: {(pred == yt).mean():.3f}   Macro F1: {f1_score(yt, pred, average='macro'):.3f}\n")
print(classification_report(yt, pred, target_names=short, digits=3))

cm = confusion_matrix(yt, pred)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(short, rotation=35, ha="right"); ax.set_yticklabels(short)
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Model said"); ax.set_ylabel("True label"); ax.set_title("v3 test results")
plt.tight_layout(); plt.savefig(WORK / "confusion_matrix_v3.png", dpi=150); plt.show()

## 8. Does the 60% threshold catch mistakes?

In the field test, wrong answers had 72-99% confidence, so the threshold didn't catch them. Here we check how many test mistakes are above 60%.

In [ ]:
wrong = pred != yt
print(f"Mistakes: {wrong.sum()} of {len(yt)}")
print(f"Mistakes with confidence above {THRESHOLD:.0%}: {(wrong & (conf >= THRESHOLD)).sum()}  (the app would show these as answers)")
print(f"Correct answers hidden by the threshold: {(~wrong & (conf < THRESHOLD)).sum()}")
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    shown = conf >= t
    acc_shown = f"{(pred[shown] == yt[shown]).mean():.1%}" if shown.any() else "n/a"
    print(f"  threshold {t:.0%}: answers shown {shown.mean():.1%}, accuracy of shown answers {acc_shown}")

## 9. Save results

In [ ]:
metrics = {
    "model": "EfficientNet-B0, ImageNet start, trained on Plant Pathology 2021 (single-label photos)",
    "class_names": CLASS_NAMES,
    "train_val_test_sizes": [len(train_df), len(val_df), len(test_df)],
    "test_accuracy": float((pred == yt).mean()),
    "test_macro_f1": float(f1_score(yt, pred, average="macro")),
    "history": [{k: float(v) for k, v in h.items()} for h in history],
}
(WORK / "class_names_v3.json").write_text(json.dumps(CLASS_NAMES, indent=2))
(WORK / "metrics_v3.json").write_text(json.dumps(metrics, indent=2))
print("Saved to", WORK, ":", sorted(os.listdir(WORK)))

## 10. (Optional) Orchard field test: v2 vs v3 on the same 9 photos

Upload a Kaggle dataset containing your 9 photos (`leaf01.jpeg` ... `leaf09.jpeg`) **and** the old model file `apple_disease_model_v2.pth`, then add it as input. This cell finds them automatically.

In [ ]:
V2_CLASSES = ["Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy"]
photos = sorted(p for p in glob.glob(f"{INPUT_ROOT}/**/leaf*", recursive=True)
                if p.lower().endswith((".jpg", ".jpeg", ".png")))
v2_hits = glob.glob(f"{INPUT_ROOT}/**/apple_disease_model_v2.pth", recursive=True)
if not photos:
    print("No orchard photos found. Skip this, or add your photos as a Kaggle dataset.")
else:
    v2 = None
    if v2_hits:
        v2 = build_model(4, False)
        v2.load_state_dict(torch.load(v2_hits[0], map_location="cpu")); v2.to(device).eval()
    # Always use the saved best v3 weights, even if earlier cells were re-run
    model = build_model(len(CLASS_NAMES), False)
    model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu")); model.to(device).eval()
    print("v2 model found:", bool(v2_hits), "| photos found:", len(photos))
    rows = []
    for path in photos:
        x = eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        row = {"photo": Path(path).stem}
        with torch.no_grad():
            for name, m, cls in [("v2", v2, V2_CLASSES), ("v3", model, CLASS_NAMES)]:
                if m is None:
                    continue
                pr = torch.softmax(m(x).float(), 1)[0].cpu().numpy()
                k = int(pr.argmax())
                label = cls[k].replace("Apple___", "") if pr[k] >= THRESHOLD else "No answer"
                row[f"{name} answer"] = label
                row[f"{name} confidence %"] = round(float(pr[k]) * 100, 1)
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(WORK / "orchard_field_test_v2_vs_v3.csv", index=False)
    print(table.to_string(index=False))